# Modul 05: Bild- und Signaldaten vorbereiten

    **Notebooktyp:** Übungs- und Bewertungsnotebook  
    **Vorlesungen dieses Moduls:** Bilddaten vorbereiten, Signale vorbereiten  
    **Erwarteter Schwierigkeitsgrad:** Leicht fortgeschritten  
    **Orientierungszeit:** etwa 100 bis 140 Minuten

    ## Überblick

    Sie erzeugen vollständig lokale Bild- und Signaldaten, untersuchen deren Formen und Wertebereiche und bauen wiederverwendbare Vorverarbeitungsschritte für Pixel, Zeitfenster, Statistik- und Frequenzmerkmale.

    ## Verwendete Vorlesungsnotebooks

    Die Aufgaben wurden aus dem Inhalt beider Vorlesungen dieses Moduls abgeleitet:

    - `ML Für Anfänger - Record_Module_05A_20260723.ipynb`
- `ML Für Anfänger - Record_Module_05B_20260723.ipynb`

    ## Colab-Kompatibilität

    Dieses Notebook ist für die kostenlose Version von Google Colab ausgelegt. Die Daten sind eingebaut, synthetisch erzeugt oder öffentlich verfügbar. Modelle und Trainingsbudgets sind bewusst klein gehalten. Führen Sie die Zellen in der vorgegebenen Reihenfolge aus.

## Lernziele

    Nach der Bearbeitung sollen Sie:

    - Kleine Bilder mit Pillow, OpenCV und NumPy laden, konvertieren und darstellen.
- Farbräume, Pixelwerte, Histogramme und grundlegende Bildoperationen untersuchen.
- Eine einheitliche Vorverarbeitung für kleine Bildsammlungen erstellen.
- Einfache Signale mit Zeitachse, Trend und Rauschen erzeugen und visualisieren.
- Fenster-, Änderungs- und Frequenzmerkmale aus Messreihen berechnen.
- Eine tabellarische Merkmalstabelle aus Signalabschnitten erstellen.

    ## Bewertete Fähigkeiten

    - RGB/BGR, Graustufen, Alpha, Normalisierung und Bildformen
- Zuschneiden, Skalieren, Glätten und Kantenerkennung
- Abtastrate, Zeitachse, gleitende Fenster und lokale Kennzahlen
- FFT, dominante Frequenz und Merkmalstabellen

## Arbeitsanweisungen

Bearbeiten Sie die Aufgaben in der angegebenen Reihenfolge. Schreiben Sie Ihren Code ausschließlich in die klar markierten Arbeitszellen. Ergänzen Sie nach jeder Aufgabe eine kurze fachliche Reflexion. Verwenden Sie das Testset nicht für Modellwahl oder Hyperparameterentscheidungen, sofern die Aufgabe dies nicht ausdrücklich als abschließenden Schritt verlangt.

- Führen Sie zuerst das gemeinsame Setup aus.
- Verändern Sie vorgegebene Splits und Seeds nur, wenn eine Aufgabe dies ausdrücklich erlaubt.
- Prüfen Sie Formen, Datentypen und Wertebereiche frühzeitig.
- Begründen Sie Modell-, Metrik- und Visualisierungsentscheidungen.
- Achten Sie auf Datenleckage und eine saubere Trennung von Training, Validierung und Test.

## Gemeinsames Setup

Führen Sie diese Zelle einmal aus, bevor Sie mit Aufgabe 1 beginnen.

In [ ]:
# Gemeinsames Setup für dieses Notebook
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import cv2
from PIL import Image
from io import StringIO

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)
np.set_printoptions(precision=3, suppress=True)
pd.set_option("display.max_columns", 50)
warnings.filterwarnings("ignore", category=FutureWarning)

# Ein synthetisches RGB-Bild vermeidet externe Dateien und Downloads.
height, width = 120, 160
synthetic_rgb = np.zeros((height, width, 3), dtype=np.uint8)
synthetic_rgb[:, :, 0] = np.linspace(20, 220, width, dtype=np.uint8)
synthetic_rgb[:, :, 1] = np.linspace(220, 40, height, dtype=np.uint8)[:, None]
synthetic_rgb[35:90, 50:115, 2] = 255
cv2.circle(synthetic_rgb, center=(35, 35), radius=18, color=(255, 255, 40), thickness=-1)

# Ein Signal mit zwei Frequenzanteilen, Trend und reproduzierbarem Rauschen.
sampling_rate_hz = 100
duration_seconds = 4
time_axis = np.arange(0, duration_seconds, 1 / sampling_rate_hz)
base_signal = (
    1.2 * np.sin(2 * np.pi * 5 * time_axis)
    + 0.45 * np.sin(2 * np.pi * 14 * time_axis)
    + 0.08 * time_axis
)
noisy_signal = base_signal + rng.normal(0, 0.18, size=time_axis.size)

print("Bildform:", synthetic_rgb.shape, synthetic_rgb.dtype)
print("Signalpunkte:", noisy_signal.size, "Abtastrate:", sampling_rate_hz, "Hz")

print("Setup abgeschlossen. Zufallsstartwert:", RANDOM_SEED)


## Aufgabe 1: Bildformate, Kanäle und Pixelwerte untersuchen

    Arbeiten Sie mit `synthetic_rgb`.

1. Erzeugen Sie ein Pillow-Bild und geben Sie Modus und Größe aus.
2. Konvertieren Sie RGB nach BGR und zurück. Prüfen Sie, ob das zurückkonvertierte Bild identisch ist.
3. Erzeugen Sie ein Graustufenbild.
4. Ergänzen Sie einen Alpha-Kanal mit einem horizontalen Transparenzverlauf.
5. Zeigen Sie RGB-, Graustufen- und RGBA-Bild in getrennten Abbildungen.

> **Hinweis:** Prüfen Sie Form, Datentyp und Wertebereich nach jeder Konvertierung.

In [ ]:
image_rgb = synthetic_rgb.copy()

# ============================================================


In [ ]:
# ============================================================
# IHR CODE HIER / YOUR CODE HERE
# ============================================================

# Schreiben Sie Ihre vollständige Lösung in diese Zelle.


### Reflexion zu Aufgabe 1

    > **Ihre Antwort:**  
    > Beschreiben Sie kurz Ihre Beobachtungen, begründen Sie wichtige Entscheidungen und nennen Sie mindestens eine mögliche Fehlerquelle.

**Pädagogischer Hinweis:** Prüfen Sie Form, Datentyp und Wertebereich nach jeder Konvertierung.

## Aufgabe 2: Normalisieren, zuschneiden, skalieren und Kanten finden

    Erstellen Sie eine Funktion `prepare_image(image_rgb, output_size=(64, 64))`, die:

1. den mittleren quadratischen Bereich des Bildes ausschneidet,
2. auf die Zielgröße skaliert,
3. Pixelwerte in `float32` und den Bereich `[0, 1]` überführt,
4. zusätzlich ein geglättetes Graustufenbild und eine Canny-Kantenkarte erzeugt,
5. Formen, Datentypen und Wertebereiche prüft.

Zeigen Sie die drei Ergebnisse in getrennten Abbildungen.

> **Hinweis:** OpenCV verwendet bei Größenangaben häufig die Reihenfolge Breite, Höhe.

In [ ]:
# ============================================================


In [ ]:
# ============================================================
# IHR CODE HIER / YOUR CODE HERE
# ============================================================

# Schreiben Sie Ihre vollständige Lösung in diese Zelle.


### Reflexion zu Aufgabe 2

    > **Ihre Antwort:**  
    > Beschreiben Sie kurz Ihre Beobachtungen, begründen Sie wichtige Entscheidungen und nennen Sie mindestens eine mögliche Fehlerquelle.

**Pädagogischer Hinweis:** OpenCV verwendet bei Größenangaben häufig die Reihenfolge Breite, Höhe.

## Aufgabe 3: Zeitachse, Abtastung und Messreihe prüfen

    Erzeugen Sie aus `time_axis` und `noisy_signal` eine Tabelle. Speichern Sie sie in einen CSV-Text im Arbeitsspeicher und laden Sie sie mit `StringIO` erneut.

1. Prüfen Sie, ob Zeitabstände konstant sind.
2. Schätzen Sie die Abtastrate aus dem Median der Zeitabstände.
3. Fügen Sie absichtlich zwei Fehlwerte und einen starken Ausreißer ein.
4. Markieren Sie die betroffenen Zeilen.
5. Zeichnen Sie das ursprüngliche und das fehlerhafte Signal in getrennten Abbildungen.

> **Hinweis:** Leiten Sie die Abtastrate aus der Zeitachse ab, statt sie nur anzunehmen.

In [ ]:
signal_table = pd.DataFrame({"time_s": time_axis, "amplitude": noisy_signal})

# ============================================================


In [ ]:
# ============================================================
# IHR CODE HIER / YOUR CODE HERE
# ============================================================

# Schreiben Sie Ihre vollständige Lösung in diese Zelle.


### Reflexion zu Aufgabe 3

    > **Ihre Antwort:**  
    > Beschreiben Sie kurz Ihre Beobachtungen, begründen Sie wichtige Entscheidungen und nennen Sie mindestens eine mögliche Fehlerquelle.

**Pädagogischer Hinweis:** Leiten Sie die Abtastrate aus der Zeitachse ab, statt sie nur anzunehmen.

## Aufgabe 4: Fenster-, Änderungs- und Frequenzmerkmale berechnen

    Schreiben Sie eine Funktion `signal_window_features(values, sampling_rate, window_size, step_size)`, die über gleitende Fenster folgende Merkmale erzeugt:

- Start- und Endindex,
- Mittelwert,
- Standardabweichung,
- RMS-Wert,
- maximaler absoluter erster Unterschied,
- dominante positive Frequenz ohne Gleichanteil.

Wenden Sie die Funktion mit Fenstergröße 100 und Schrittweite 50 auf `noisy_signal` an. Visualisieren Sie die dominante Frequenz je Fenster.

> **Hinweis:** Entfernen Sie den Fenstermittelwert, bevor Sie die stärkste positive Frequenz suchen.

In [ ]:
# ============================================================


In [ ]:
# ============================================================
# IHR CODE HIER / YOUR CODE HERE
# ============================================================

# Schreiben Sie Ihre vollständige Lösung in diese Zelle.


### Reflexion zu Aufgabe 4

    > **Ihre Antwort:**  
    > Beschreiben Sie kurz Ihre Beobachtungen, begründen Sie wichtige Entscheidungen und nennen Sie mindestens eine mögliche Fehlerquelle.

**Pädagogischer Hinweis:** Entfernen Sie den Fenstermittelwert, bevor Sie die stärkste positive Frequenz suchen.

## Aufgabe 5: Integrationsaufgabe: Kleine Bild- und Signal-Batches vorbereiten

    Erstellen Sie zwei wiederverwendbare Ausgaben.

**Bildteil:** Erzeugen Sie aus drei Varianten von `synthetic_rgb` einen Batch normalisierter Bilder der Form `(3, 64, 64, 3)`. Verwenden Sie Original, horizontal gespiegeltes Bild und eine dunklere Variante.

**Signalteil:** Erzeugen Sie sechs Signale, drei mit ungefähr 5 Hz und drei mit ungefähr 12 Hz. Berechnen Sie pro vollständigem Signal dieselben Statistik- und Frequenzmerkmale und erstellen Sie eine Merkmalstabelle mit Zielspalte `signal_class`.

Prüfen Sie Formen, Datentypen, Wertebereiche und Klassenverteilung.

> **Hinweis:** Die Batchdimension zählt Beispiele, nicht Bildkanäle oder Zeitpunkte.

In [ ]:
# ============================================================


In [ ]:
# ============================================================
# IHR CODE HIER / YOUR CODE HERE
# ============================================================

# Schreiben Sie Ihre vollständige Lösung in diese Zelle.


### Reflexion zu Aufgabe 5

    > **Ihre Antwort:**  
    > Beschreiben Sie kurz Ihre Beobachtungen, begründen Sie wichtige Entscheidungen und nennen Sie mindestens eine mögliche Fehlerquelle.

**Pädagogischer Hinweis:** Die Batchdimension zählt Beispiele, nicht Bildkanäle oder Zeitpunkte.

## Abschluss und Selbstkontrolle

Prüfen Sie vor der Abgabe, ob alle Arbeitszellen ausgefüllt sind, das Notebook von oben nach unten ohne unerwartete Fehler läuft, alle Diagramme beschriftet sind und jede Reflexion Ihre Beobachtungen sowie mindestens eine mögliche Fehlerquelle enthält.

- Alle Aufgaben und Unterpunkte wurden bearbeitet.
- Verwendete Seeds und Datenpartitionen sind nachvollziehbar.
- Testdaten wurden nicht vorzeitig für Entscheidungen genutzt.
- Ergebnisse werden vorsichtig und fachlich begründet interpretiert.
- Es gibt keine hardcodierten lokalen Dateipfade oder privaten Zugangsdaten.